In [22]:
import math

In [4]:
def f(x):
    return 3*x**2 - 4*x + 5

In [5]:
f(2.0)

9.0

In [6]:
h = 0.0001
x = 2.0

slope = (f(x+h) - f(x)) / h
slope

8.000300000023941

In [7]:
for h in [0.1 , 0.01, 0.001, 0.0001, 0.00001, 0.000001]:
    slope = (f(x+h) - f(x)) / h
    print(h, slope)

0.1 8.3
0.01 8.02999999999976
0.001 8.003000000000426
0.0001 8.000300000023941
1e-05 8.000030000054892
1e-06 8.000003001384925


In [8]:
def f(a, b, c):
    return a*b + c

a, b, c = 2.0 , -3.0, 10.0
h = 0.0001

# gradient of a
grad_a = (f(a+h, b, c) - f(a, b, c)) / h

# gradient of b
grad_b = (f(a, b+h, c) - f(a, b, c)) / h

# gradient of c
grad_c = (f(a, b, c+h) - f(a, b, c)) / h

print("grad_a:", grad_a)
print("grad_b:", grad_b)
print("grad_c:", grad_c)

grad_a: -3.000000000010772
grad_b: 2.0000000000042206
grad_c: 0.9999999999976694


In [11]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')
        return out

In [12]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c
print(d)
print(d._prev)
print(d._op)

Value(data=4.0)
{Value(data=10.0), Value(data=-6.0)}
+


In [13]:
print(e)
print(e._prev)
print(e._op)

Value(data=-6.0)
{Value(data=-3.0), Value(data=2.0)}
*


In [27]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1 * out.grad
            other.grad += 1 * out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [18]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c

d.grad = 1.0           # we start from the root, the derivative according to itself = 1
d._backward()          # distribute the gradient of d to e and c
e._backward()          # distribute the gradient of e to a and b

print("a.grad:", a.grad)
print("b.grad:", b.grad)
print("c.grad:", c.grad)
print("e.grad:", e.grad)

a.grad: -3.0
b.grad: 2.0
c.grad: 1.0
e.grad: 1.0


In [19]:
def backward(self):
    topo = []
    visited = set()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build_topo(child)
            topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
        node._backward()

Value.backward = backward

In [20]:
a = Value(2.0)
b = Value(-3.0)
c = Value(10.0)
e = a * b
d = e + c

d.backward()

print("a.grad:", a.grad)
print("b.grad:", b.grad)
print("c.grad:", c.grad)

a.grad: -3.0
b.grad: 2.0
c.grad: 1.0


In [21]:
a = Value(3.0)
b = a + a

b.backward()

print("a.grad:", a.grad)

a.grad: 2.0


In [25]:
x = Value(0.0)
y = x.tanh()
print(y)

Value(data=0.0)


In [28]:
# inputs
x1 = Value(2.0)
x2 = Value(0.0)

# weights
w1 = Value(-3.0)
w2 = Value(1.0)

# bias
b = Value(6.8813735870195432)

# forward pass
x1w1 = x1 * w1
x2w2 = x2 * w2
x1w1x2w2 = x1w1 + x2w2
n = x1w1x2w2 + b          # This value is usually called as "pre-activation"
o = n.tanh()              # the version that has passed through activation

print("n:", n)
print("o:", o)

n: Value(data=0.8813735870195432)
o: Value(data=0.7071067811865476)


In [29]:
o.backward()

print("x1.grad:", x1.grad)
print("x2.grad:", x2.grad)
print("w1.grad:", w1.grad)
print("w2.grad:", w2.grad)
print("b.grad:", b.grad)

x1.grad: -1.4999999999999996
x2.grad: 0.4999999999999999
w1.grad: 0.9999999999999998
w2.grad: 0.0
b.grad: 0.4999999999999999
